In [ ]:
import numpy as np

def CRR(S0, K, T, r, sd, N):
    dt = T / N
    u = np.exp(sd * np.sqrt(dt))
    d = np.exp(-sd * np.sqrt(dt))
    q = (np.exp(r * dt) - d) / (u - d)
    df = np.exp(-r * dt)

    
    S = S0 * d ** (np.arange(N, -1, -1)) * u ** (np.arange(0, N + 1, 1))
    C = np.maximum(K - S, 0)  

    S_tree = {}
    C_tree = {}

    for i in np.arange(N - 1, -1, -1):
        S = S0 * d ** (np.arange(i, -1, -1)) * u ** (np.arange(0, i + 1, 1))
        continuation = df * (q * C[1:i + 2] + (1 - q) * C[0:i + 1])
        exercise = np.maximum(K - S, 0)
        C = np.maximum(continuation, exercise)

        
        if i == 1:
            C_tree[1] = C.copy()
            S_tree[1] = S.copy()
        elif i == 2:
            C_tree[2] = C.copy()
            S_tree[2] = S.copy()

    # Delta ≈ (C_up - C_down) / (S_up - S_down)
    C_up = C_tree[1][0]
    C_down = C_tree[1][1]
    S_up = S_tree[1][0]
    S_down = S_tree[1][1]
    delta = (C_up - C_down) / (S_up - S_down)

    # Gamma ≈ ((C_uu - C_ud) / (S_uu - S_ud) - (C_ud - C_dd) / (S_ud - S_dd)) / ((S_uu - S_dd)/2)
    C_uu = C_tree[2][0]
    C_ud = C_tree[2][1]
    C_dd = C_tree[2][2]
    S_uu = S_tree[2][0]
    S_ud = S_tree[2][1]
    S_dd = S_tree[2][2]
    gamma = ((C_uu - C_ud) / (S_uu - S_ud) - (C_ud - C_dd) / (S_ud - S_dd)) / ((S_uu - S_dd) / 2)

    return round(C[0], 3), round(delta, 4), round(gamma, 4)



price, delta, gamma = CRR(S0=40, K=44, T=1, r=0.06, sd=0.2, N=10**3)
print("Option Price:", price)
print("Delta:", delta)
print("Gamma:", gamma)


Option Price: 4.663
Delta: -0.6657
Gamma: 0.0765
